# PPE Guard — upper YOLO26m detector

This notebook trains **only the upper detector** for goggles and wrist accessories.
It produces **upper.pt**. Run the other notebook separately in the other account.

Attach the updated **kaggle-training.zip** as a private Kaggle dataset in this account.
Enable a GPU accelerator and Internet, then run cells in order. The same data ZIP
works for both accounts; this notebook copies only its detector's data.
Expected data: **2,281 images — 1,379 train / 542 validation / 360 test**.
Training is 100 epochs with early stopping, 960-pixel input, batch 2, and YOLO26m only.
If GPU memory runs out, lower BATCH to 1. Download the model and reports before ending the session.
Training results alone do not establish factory accuracy.


In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "ultralytics==8.4.188",
        "polars==1.35.2",
        "opencv-python==4.12.0.88",
    ],
    check=True,
)
import torch  # noqa: E402

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import json
import shutil
import zipfile
import yaml

VIEW = "upper"
project = Path(f"/kaggle/working/ppe-guard-{VIEW}").resolve()
project.mkdir(parents=True, exist_ok=True)
archives = list(Path("/kaggle/input").rglob("kaggle-training.zip"))
if archives:
    assert len(archives) == 1, "Attach only one kaggle-training.zip version"
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if not (
                member.filename.startswith(f"data/ppe/{VIEW}/")
                or member.filename in ["tools/train.py", "data/ppe/split-manifest.json"]
            ):
                continue
            target = (project / member.filename).resolve()
            assert target.is_relative_to(project), "Unsafe archive member"
            archive.extract(member, project)
else:
    # Kaggle can expose an uploaded ZIP as already extracted files.
    candidates = [
        p
        for p in Path("/kaggle/input").rglob("data.yaml")
        if p.as_posix().endswith(f"/data/ppe/{VIEW}/data.yaml")
    ]
    assert len(candidates) == 1, "Attach the updated private Kaggle training dataset"
    source_root = candidates[0].parents[3]
    shutil.copytree(
        source_root / f"data/ppe/{VIEW}", project / f"data/ppe/{VIEW}", dirs_exist_ok=True
    )
    (project / "tools").mkdir(exist_ok=True)
    shutil.copy2(source_root / "tools/train.py", project / "tools/train.py")
    shutil.copy2(
        source_root / "data/ppe/split-manifest.json", project / "data/ppe/split-manifest.json"
    )

data_file = project / f"data/ppe/{VIEW}/data.yaml"
spec = yaml.safe_load(data_file.read_text())
expected_names = {
    "upper": ["goggles", "no_goggles", "empty_wrist", "not_empty_wrist"],
    "lower": ["SAFETY_SHOE", "NOT_SAFETY_SHOE"],
}
names = list(spec["names"].values()) if isinstance(spec["names"], dict) else spec["names"]
assert names == expected_names[VIEW], "Wrong class mapping"
manifest = json.loads((project / "data/ppe/split-manifest.json").read_text())
assert len(manifest["images"]) == 2281, "Dataset is outdated; upload the updated ZIP"
for split, expected in [("train", 1379), ("valid", 542), ("test", 360)]:
    images = {p.stem for p in (data_file.parent / "images" / split).glob("*.jpg")}
    labels = {p.stem for p in (data_file.parent / "labels" / split).glob("*.txt")}
    assert images == labels and len(images) == expected, f"Incomplete {split} dataset"
print(VIEW, names, "— 2,281 images ready")

## Train this detector

Each run has its own output folder, so the exported checkpoint comes from this exact run.
Validation runs during training. The test set is not used to choose the best epoch.


In [ ]:
import uuid

IMAGE_SIZE = 960
BATCH = 2
EPOCHS = 100
weights = None
run_output = project / "runs" / uuid.uuid4().hex
subprocess.run(
    [
        sys.executable,
        str(project / "tools/train.py"),
        "train",
        VIEW,
        "--data",
        str(data_file),
        "--model",
        "yolo26m.pt",
        "--imgsz",
        str(IMAGE_SIZE),
        "--batch",
        str(BATCH),
        "--epochs",
        str(EPOCHS),
        "--output",
        str(run_output),
    ],
    cwd=project,
    check=True,
)
weights = run_output / VIEW / "weights/best.pt"
assert weights.is_file(), "Training did not produce a best checkpoint"
print("Completed:", weights)

## Optional final test

Leave this off while trying training settings. After reviewing validation failures and
choosing the final run, set RUN_HELDOUT_TEST to True and run once. Do not tune against
these test results. Independent factory recordings are still needed before deployment.


In [ ]:
RUN_HELDOUT_TEST = False
if RUN_HELDOUT_TEST:
    assert weights is not None and weights.is_file(), "Complete training first"
    subprocess.run(
        [
            sys.executable,
            str(project / "tools/train.py"),
            "evaluate",
            VIEW,
            "--data",
            str(data_file),
            "--weights",
            str(weights),
            "--imgsz",
            str(IMAGE_SIZE),
            "--batch",
            str(BATCH),
            "--output",
            str(run_output),
        ],
        cwd=project,
        check=True,
    )

## Download upper.pt and reports

Download **upper.pt** from the links below or from notebook Output.
The companion ZIP includes the model, training/validation plots and any final-test reports.
After both notebooks finish, the app uses upper.pt and lower.pt in its models folder.


In [ ]:
import os
from IPython.display import FileLink, display

assert weights is not None and weights.is_file(), "Complete training first"
export = project.parent / f"{VIEW}.pt"
shutil.copy2(weights, export)
reports = project.parent / f"ppe-{VIEW}-model-and-reports.zip"
with zipfile.ZipFile(reports, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(export, export.name)
    for path in run_output.rglob("*"):
        if path.is_file() and path.suffix.lower() in [".png", ".jpg", ".csv", ".json", ".yaml"]:
            archive.write(path, "reports/" + path.relative_to(run_output).as_posix())
os.chdir(project.parent)
display(FileLink(export.name))
display(FileLink(reports.name))